In [1]:
import pandas as pd
import math
import heapq
from collections import defaultdict

In [5]:
df=pd.read_csv("cities.csv")
df.head(20)
df.head()

,City,Lat,Long,country,iso2,State
0,Abohar,30.144533,74.195520,India,IN,Punjab
1,Adilabad,19.400000,78.310000,India,IN,Telangana
2,Agartala,23.836049,91.279386,India,IN,Tripura
3,Agra,27.187935,78.003944,India,IN,Uttar Pradesh
4,Ahmadnagar,19.094571,74.738432,India,IN,Maharashtra


In [15]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371

    # Convert to radians FIRST
    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (math.sin(dlat/2)**2 +
         math.cos(lat1) * math.cos(lat2) *
         math.sin(dlon/2)**2)

    # Fix floating error (VERY IMPORTANT)
    a = min(1, max(0, a))

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return int(R * c)

In [16]:
import math
data = []

for i in range(len(df)):
    for j in range(len(df)):
        if i != j:
            City1 = df.iloc[i]['City']
            City2 = df.iloc[j]['City']

            d = haversine(
                df.iloc[i]['Lat'], df.iloc[i]['Long'],
                df.iloc[j]['Lat'], df.iloc[j]['Long']
            )

            data.append([City1, City2, d])   # ✅ FIX HERE

dist_df = pd.DataFrame(data, columns=["City1", "City2", "distance"])
dist_df.head()

,City1,City2,distance
0,Abohar,Adilabad,1264
1,Abohar,Agartala,1829
2,Abohar,Agra,496
3,Abohar,Ahmadnagar,1229
4,Abohar,Ahmedabad,807


In [17]:
from collections import defaultdict

def build_graph(df):
    graph = defaultdict(list)

    for _, row in df.iterrows():
        city1 = row['City1']
        city2 = row['City2']
        dist = row['distance']

        graph[city1].append((city2, dist))

    return graph

graph = build_graph(dist_df)

In [18]:
import heapq

def dijkstra(graph, start):
    pq = [(0, start)]
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    parent = {node: None for node in graph}

    while pq:
        curr_dist, curr_node = heapq.heappop(pq)

        for neighbor, weight in graph[curr_node]:
            distance = curr_dist + weight

            if distance < distances[neighbor]:
                distances[neighbor] = distance
                parent[neighbor] = curr_node
                heapq.heappush(pq, (distance, neighbor))

    return distances, parent

In [19]:
def get_path(parent, city):
    path = []
    while city:
        path.append(city)
        city = parent[city]
    return path[::-1]

In [27]:
start = input("Enter source city: ")
end = input("Enter destination city: ")

if start not in graph or end not in graph:
    print(" City not found")
else:
    distances, parent = dijkstra(graph, start)

    path = get_path(parent, end)

    print(f"\n Shortest Distance: {distances[end]} km")
    print(f" Path: {' → '.join(path)}")

    # Show map
    m = plot_path(path, city_coords)
    m

Enter source city:  Agra
Enter destination city:  Hyderabad



 Shortest Distance: 1090 km
 Path: Agra → Adilabad → Hyderabad


In [22]:
city_coords = {}

for _, row in df.iterrows():
    city_coords[row['City']] = (row['Lat'], row['Long'])

In [ ]:
import sys
!{sys.executable} -m pip install folium --user

In [28]:
import folium

def plot_path(path, city_coords):
    m = folium.Map(location=[22.5, 78.9], zoom_start=5)


    for city in path:
        lat, lon = city_coords[city]
        folium.Marker([lat, lon], popup=city).add_to(m)

    
    points = [city_coords[city] for city in path]
    folium.PolyLine(points, color="blue", weight=5).add_to(m)

    return m